# Wi-Fi 7 multi-link EDCA — result figures

Reproduces the three result figures and the running-time table from the stored
measurements. Nothing is re-simulated here: every number was produced by the
analytical model in the main repository and written to the JSON files loaded
below, so this notebook only reads and plots.

**Data files needed** (upload them, or put them in a folder named `data`
next to the notebook):

| file | used by |
|---|---|
| `pp1_budget_methods.json` | Fig. 1, running-time table |
| `pp1_qos_methods.json` | Fig. 2, running-time table |
| `gnn_train_curve.json` | Fig. 3 |
| `pp1_paper_figs.json` | Fig. 3, running-time table |
| `pipeline_ngen1.json` | running-time table |
| `pp1_anytime.json` | Fig. 4 |

Three methods appear throughout, and they are cumulative — each adds one
component to the one above it:

| key | figure label | what it is |
|---|---|---|
| `ga` | GA baseline (Yi et al.) | genetic search on the analytical model |
| `ga_gnn` | `+` evaluation GNN $g_\phi$ | GNN predicting $(\log_{10} c_i,\ \theta_i)$; **ranks** candidates so the analytical model is called less often |
| `ga_gnn_policy` | `+` proposal GNN $\pi_\psi$ | policy GNN mapping a scenario to a distribution over gene levels; **seeds** the initial population |

Run the cells in order.

In [ ]:
# --- Cell 1: locate the data -------------------------------------------------
import json, os, glob

NEEDED = ["pp1_budget_methods.json", "pp1_qos_methods.json",
          "gnn_train_curve.json", "pp1_paper_figs.json",
          "pipeline_ngen1.json", "pp1_anytime.json"]

def find_data_dir():
    """First directory that holds all five files. Falls back to an upload."""
    for d in ("data", ".", "/content/data", "/content"):
        if all(os.path.exists(os.path.join(d, f)) for f in NEEDED):
            return d
    return None

DATA_DIR = find_data_dir()
if DATA_DIR is None:
    try:                                  # Colab: ask for the files directly
        from google.colab import files
        print("Upload the five JSON files (Ctrl-click to select all):")
        files.upload()
        DATA_DIR = find_data_dir()
    except ImportError:
        pass

if DATA_DIR is None:
    missing = [f for f in NEEDED
               if not any(glob.glob(os.path.join(d, f))
                          for d in ("data", ".", "/content/data", "/content"))]
    raise FileNotFoundError("still missing: %s" % ", ".join(missing))

print("data dir:", os.path.abspath(DATA_DIR))
for f in NEEDED:
    print("  %-28s %8.1f kB" % (f, os.path.getsize(os.path.join(DATA_DIR, f)) / 1e3))

In [ ]:
# --- Cell 2: plotting style --------------------------------------------------
# Matched to an IEEEtran two-column layout: figures are drawn at the width they
# are printed at, so the point sizes below are read literally on the page.
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

COL, PAGE = 3.50, 7.16          # \columnwidth and \textwidth, in inches
PT_TICK, PT_LABEL, PT_TITLE, PT_LEGEND, PT_NOTE = 7.0, 8.0, 8.0, 7.0, 6.5

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Nimbus Roman No9 L", "Liberation Serif",
                   "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": PT_TICK,
    "axes.labelsize": PT_LABEL, "axes.titlesize": PT_TITLE,
    "axes.titleweight": "normal", "axes.titlepad": 3.0, "axes.labelpad": 2.0,
    "axes.linewidth": 0.6, "axes.edgecolor": "black",
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": True, "axes.spines.right": True,
    "grid.color": "#CCCCCC", "grid.linewidth": 0.4, "grid.linestyle": "-",
    "xtick.labelsize": PT_TICK, "ytick.labelsize": PT_TICK,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "xtick.major.size": 2.6, "ytick.major.size": 2.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.pad": 2.0, "ytick.major.pad": 2.0,
    "legend.frameon": True, "legend.edgecolor": "black",
    "legend.framealpha": 1.0, "legend.fancybox": False,
    "legend.fontsize": PT_LEGEND, "legend.borderpad": 0.35,
    "legend.labelspacing": 0.28, "legend.handlelength": 2.0,
    "legend.handletextpad": 0.45, "legend.columnspacing": 1.1,
    "lines.linewidth": 1.1, "lines.markersize": 3.4,
    "lines.markeredgewidth": 0.5,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "savefig.bbox": "tight",
    "savefig.pad_inches": 0.01, "figure.dpi": 110,
})

# Okabe-Ito, so the three methods stay distinguishable under colour blindness
# and in greyscale print. One (colour, linestyle, marker) triple per method,
# fixed across every figure.
STYLE = {"ga":            {"c": "#000000", "ls": "-",  "m": "o"},
         "ga_gnn":        {"c": "#0072B2", "ls": "--", "m": "s"},
         "ga_gnn_policy": {"c": "#D55E00", "ls": "-",  "m": "^"}}
C_DEF, RED, GREY = "#7F7F7F", "#C0392B", "#555555"

LAB = {"ga": "GA baseline (Yi et al.)",
       "ga_gnn": r"$+$ evaluation GNN $g_\phi$ (ranks candidates)",
       "ga_gnn_policy": r"$+$ proposal GNN $\pi_\psi$ (seeds the population)"}
LAB_S = {"ga": "GA baseline (Yi et al.)",
         "ga_gnn": r"$+\,g_\phi$ (evaluation GNN)",
         "ga_gnn_policy": r"$+\,\pi_\psi$ (proposal GNN)"}
ORDER = ["ga", "ga_gnn", "ga_gnn_policy"]

# The tables below are read as plain text, not typeset by matplotlib, so they
# need names with no math delimiters; the LaTeX export needs the opposite.
LAB_TXT = {"ga": "GA baseline (Yi et al.)",
           "ga_gnn": "  + evaluation GNN (g_phi)",
           "ga_gnn_policy": "  + proposal GNN (pi_psi)"}
LAB_TEX = {"ga": r"GA baseline~\cite{Yi2025}",
           "ga_gnn": r"\;\; $+$ evaluation GNN $g_\phi$",
           "ga_gnn_policy": r"\;\; $+$ proposal GNN $\pi_\psi$"}
DEPLOY_TXT = "Pipeline as deployed (Npop=160, Ngen=1)"
DEPLOY_TEX = (r"Pipeline as deployed "
              r"($N_{\mathrm{pop}} = 160$, $N_{\mathrm{gen}} = 1$)")

def legend_below(fig, ax, ncol=3, y=-0.16, handles=None, labels=None):
    """One shared legend under the figure, so it covers no curve."""
    if handles is None:
        handles, labels = ax.get_legend_handles_labels()
    return fig.legend(handles, labels, loc="lower center", ncol=ncol,
                      bbox_to_anchor=(0.5, y), fontsize=PT_LEGEND,
                      frameon=True, edgecolor="black", framealpha=1.0,
                      fancybox=False)

print("style ready")

In [ ]:
# --- Cell 3: load ------------------------------------------------------------
def load(name):
    with open(os.path.join(DATA_DIR, name), encoding="utf-8") as fh:
        return json.load(fh)

BM = load("pp1_budget_methods.json")   # eps sweep x budget x method
QM = load("pp1_qos_methods.json")      # per-AC QoS, four configurations
TC = load("gnn_train_curve.json")      # evaluation-GNN training curve
D  = load("pp1_paper_figs.json")       # proposal-GNN training curve + pipeline
NG1 = load("pipeline_ngen1.json")["160x1"]   # deployed pipeline, N_gen = 1
AT = load("pp1_anytime.json")          # objective against compute, 20 seeds

print("budgets            :", BM["budgets"])
print("eps_1 grid         :", BM["eps1"])
print("restarts per point :", BM["n_restart"])
print("QoS configurations :", list(QM["configs"]))
print("anytime run        : %d seeds, no early stopping" % AT["n_seed"])
print("evaluation-GNN curve: %d checkpoints, %d epochs"
      % (len(TC["curve"]), TC["epochs"]))
print("probe pool         : %d candidates, %.1f%% feasible, ceiling F = %.3f"
      % (TC["pool"]["n"], 100 * TC["pool"]["frac_feasible"],
         TC["pool"]["oracle_topv"]))

## Fig. 1 — behaviour when the tightest threshold moves

All three methods at a common budget ($N_{pop} = N_{gen} = 120$), best of ten
independent restarts per point.

Each curve is the **mean** over the ten restarts, with $\pm$ one standard
deviation shaded in (a). The best of the ten answers "what can this method
reach"; the mean answers "what does one run of it return".

Budgets are **matched on analytical-model calls**, not on $(N_{pop}, N_{gen})$:
the same grid point costs the baseline $12{,}426$ calls against $1{,}070$ and
$474$ for the two network configurations. The three curves here spend $526$,
$533$ and $474$, within 12% of each other, and the legend carries each cost.
Each point is **40 restarts**, not 10: at 10 the standard error of the mean is
1--3 objective points, enough to invent dips that are not there. Set every entry
of `SWEEP_BUDGET` to `"120x120"` in the next cell for the
equal-$(N_{pop}, N_{gen})$ version, where the baseline catches up.

**Panel (a) is bounded; panel (b) is not.** The objective
$F = \sum_i -\log_{10} P_{\mathrm{loss},i}$ does not contain $\varepsilon$,
which enters only through the constraints $\Pr(D_i \ge D_{\max,i}) < \varepsilon_i$.
Relaxing $\varepsilon_1$ therefore only enlarges the feasible set, so
$F^\star(\varepsilon_1)$ **cannot fall** — a drop in (a) is a convergence failure
of the solver, not a property of the problem. $\sum_i \theta_i$ is not the
objective, so no such bound applies in (b): the configuration that maximises $F$
is free to spend reliability margin where the constraints do not ask for it, and
a dip there is a property of the solution. The bound is on the optimum, which
the best of ten estimates, not on the mean; the cell prints the check for both,
and the baseline mean does dip.

In [ ]:
# --- Cell 4: Fig. 1 ----------------------------------------------------------
# Budget per method. NOT a common (Npop, Ngen): that costs the three methods
# 12,426, 1,070 and 474 calls to the analytical model, which is the scarce
# quantity. These three are matched on THAT -- 526, 533 and 474 calls -- so the
# figure compares what each does with the same compute. Each is 40 restarts per
# point, not 10; at 10 the standard error of the mean is 1-3 objective points,
# enough to invent dips. Set all three to "120x120" for the equal-(Npop, Ngen)
# version, where the baseline catches up.
SWEEP_BUDGET = {"ga": "16x9r40", "ga_gnn": "40x40r40",
                "ga_gnn_policy": "120x120r40"}


def fig_sweep(budgets=None, save="fig1_sweep_methods.png"):
    budgets = budgets or SWEEP_BUDGET
    idx = {k: BM["methods"][k]["budget"].index(v) for k, v in budgets.items()}
    eps1 = np.array(BM["eps1"], float)
    tgt = np.array(BM["target_sum_theta"], float)

    fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.30))
    for k in ORDER:
        st, e, b = STYLE[k], BM["methods"][k], idx[k]
        lab = "%s, %s calls" % (LAB_S[k], "{:,}".format(round(e["calls"][b])))
        # The band is drawn in (a) only. There the spread IS the finding: the
        # seeded pipeline has none and the two unseeded methods have a lot. In
        # (b) the plotted quantity is not the objective, the three curves run
        # within two points of each other, and three overlapping bands would
        # hide both them and the target line.
        panels = ((axes[0], e["mean"][b], e["std"][b]),
                  (axes[1], e["sum_theta_mean"][b], None))
        for ax, mu, sd in panels:
            mu = np.array(mu, float)
            if sd is not None:
                sd = np.array(sd, float)
                ax.fill_between(eps1, mu - sd, mu + sd, color=st["c"],
                                alpha=0.11, lw=0, zorder=1)
            ax.plot(eps1, mu, ls=st["ls"], marker=st["m"], color=st["c"],
                    ms=4.0, mec="black", mew=0.4, zorder=3, label=lab)

    ax = axes[0]
    ax.axhline(50, color=GREY, lw=0.6, ls=(0, (1, 2)))
    # Below the line: every curve reaches the ceiling at the loose end, so a
    # label above it lands on the frame and the panel title.
    # Left, not right: the pipeline curve reaches the ceiling from about
    # 1e-7 onwards and a right-hand label lands on it.
    ax.text(0.02, 49.4, "ceiling $F = 50$", transform=ax.get_yaxis_transform(),
            ha="left", va="top", fontsize=PT_NOTE, color=GREY)
    ax.set_ylabel("Objective $F$")
    ax.set_title("(a) Fitness value, mean of $10$ restarts")

    ax = axes[1]
    ax.plot(eps1, tgt, ":", color="black", lw=1.0, label=r"Target $\varepsilon$")
    ax.set_ylabel(r"$\sum_i \theta_i$")
    ax.set_title("(b) Sum of reliability indices")

    for ax in axes:
        ax.set_xscale("log")
        ax.set_xlabel(r"$\varepsilon_1$")

    fig.tight_layout(pad=0.3)
    h, l = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    h.append(h2[-1]); l.append(l2[-1])
    legend_below(fig, axes[0], ncol=4, handles=h, labels=l)
    if save:
        fig.savefig(save, dpi=600)
        print("saved", save)
    plt.show()

    # The monotonicity bound applies to the OPTIMUM, which the best-of-ten
    # estimates; the mean is a property of the solver at a fixed budget and
    # carries no such guarantee. Both are reported.
    mono = lambda f: all(f[i] <= f[i + 1] + 1e-9 for i in range(len(f) - 1))
    print()
    for k in ORDER:
        e, b = BM["methods"][k], idx[k]
        mu, sd, bs = e["mean"][b], e["std"][b], e["best"][b]
        print("  %-14s %-11s %6.0f calls, n=%d"
              % (k, budgets[k], e["calls"][b], e["n_restart"][b]))
        print("  %-14s mean %s  (sd %.2f-%.2f)  non-decreasing %s"
              % ("", " ".join("%5.2f" % v for v in mu), min(sd), max(sd),
                 mono(mu)))
        print("  %-14s best %s  non-decreasing %s"
              % ("", " ".join("%5.2f" % v for v in bs), mono(bs)))

fig_sweep()

## Fig. 2 — QoS of the returned configuration

Delay-violation probability against the per-category target, and packet-loss
probability, for the default IEEE 802.11e table and all three methods.

The configuration plotted for each method is its **best of ten** independent
restarts at a common budget. A single run would mostly plot seed noise: on this
scenario the spread between restarts (GA best $49.70$ against median $43.11$) is
wider than the spread between methods.

Panel (b) is worth drawing only because $g_\phi$ alone is present. With just the
baseline and the full pipeline, every optimised configuration sits on the
$10^{-10}$ reporting floor and the panel separates nothing.

In [ ]:
# --- Cell 5: Fig. 2 ----------------------------------------------------------
def fig_qos(save="fig2_qos_methods.png"):
    acs = QM["ac"]
    eps = np.array(QM["eps"], float)
    x = np.arange(len(acs))
    w = 0.20
    # Log bars need a floor below the data. The default table puts AC2 at
    # 1e-16, thirteen decades below anything else here; giving the axis room
    # for it would compress every comparison that matters into the top fifth of
    # the frame, so the floor sits just below the optimised values instead and
    # the bars that fall through it are declared on the panel.
    LO_A, LO_B = 1e-11, 1e-12
    series = [("default", C_DEF, "Default EDCA")] + \
             [(k, STYLE[k]["c"], LAB_S[k]) for k in ORDER]

    fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.35))
    for i, (k, c, lab) in enumerate(series):
        off = (i - 1.5) * w
        va = np.maximum(np.array(QM["configs"][k]["violation"], float), LO_A)
        pl = np.maximum(np.array(QM["configs"][k]["p_loss"], float), LO_B)
        axes[0].bar(x + off, va, w, color=c, edgecolor="black", lw=0.4,
                    label=lab, zorder=3)
        axes[1].bar(x + off, pl, w, color=c, edgecolor="black", lw=0.4,
                    label=lab, zorder=3)

    ax = axes[0]
    ax.plot(x, eps, "_", color=RED, ms=15, mew=1.4, ls="none", zorder=5,
            label=r"Target $\varepsilon_i$")
    ax.set_ylim(LO_A, 1e3)
    ax.set_ylabel(r"$\Pr(D \geq D_{\max})$")
    ax.set_title("(a) Delay violation probability")
    ax.text(0.02, 0.96, r"default-table bars below $10^{-11}$ are clipped",
            transform=ax.transAxes, va="top", fontsize=PT_NOTE, color=GREY)

    ax = axes[1]
    ax.axhline(1e-10, color=GREY, lw=0.7, ls=(0, (3, 2)), zorder=4)
    # In the frame corner, not on the line: the floor cuts through the bars of
    # every category, so any in-place label sits on top of one.
    ax.text(0.02, 0.96, r"dashed: reporting floor $10^{-10}$",
            transform=ax.transAxes, va="top", fontsize=PT_NOTE, color=GREY)
    ax.set_ylim(LO_B, 1e2)
    ax.set_ylabel(r"$P_{\mathrm{loss}}$")
    ax.set_title("(b) Packet loss probability")

    for ax in axes:
        ax.set_yscale("log")
        ax.set_xticks(x); ax.set_xticklabels(acs)
        ax.set_xlabel("Access category")
        ax.grid(True, axis="y", which="major", lw=0.35, color="#D5D5D5")

    fig.tight_layout(pad=0.3)
    legend_below(fig, axes[0], ncol=5, y=-0.17)
    if save:
        fig.savefig(save, dpi=600)
        print("saved", save)
    plt.show()

    print("\nfeasible at every category?")
    for k, _, lab in series:
        v = np.array(QM["configs"][k]["violation"], float)
        bad = [acs[i] for i in range(len(acs)) if v[i] >= eps[i]]
        print("  %-14s %s" % (k, "yes" if not bad else "no -- " + ", ".join(bad)))

fig_qos()

## Fig. 3 — training convergence of both networks

The two networks are trained on different problems and **cannot share an x
axis**: one counts cross-entropy iterations, the other passes over a labelled
set. The **y axis is shared**, and that is the point — each network is scored by
the objective $F$ its own output achieves under the analytical model.

- $\pi_\psi$ emits a configuration, so it is scored by the exact $F$ of that
  configuration.
- $g_\phi$ emits an **ordering**, so it is scored the way the pipeline uses it:
  the mean exact $F$ of the $V = 8$ candidates it forwards for exact evaluation,
  out of a fixed pool of 1,560 configurations scored once in advance. A
  regression error (RMSE) does not belong on this axis; the ranking it induces
  does. The dotted ceiling is what a perfect ordering of the same pool scores.

In [ ]:
# --- Cell 6: Fig. 3 ----------------------------------------------------------
def fig_training(save="fig3_gnn_training.png"):
    fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.45), sharey=True)

    # (a) proposal network: exact F of the configuration it emits
    e1 = D["e1"]
    step = np.array(e1["step"], float)
    exact = np.array(e1["exact"], float)
    ok = np.array(e1["feasible"], bool)
    cp = STYLE["ga_gnn_policy"]["c"]

    ax = axes[0]
    ax.axhline(D["ga_fit_max"], color=STYLE["ga"]["c"], ls="-", lw=0.9)
    ax.axhline(D["ga_fit_med"], color=STYLE["ga"]["c"], ls="--", lw=0.9)
    ax.axhspan(D["ga_fit_med"], D["ga_fit_max"], color=STYLE["ga"]["c"],
               alpha=0.07, zorder=0)
    ax.text(0.98, D["ga_fit_max"] + 1.0, "GA best of $10$ seeds",
            transform=ax.get_yaxis_transform(), ha="right", va="bottom",
            fontsize=PT_NOTE, color=STYLE["ga"]["c"])
    ax.text(0.98, D["ga_fit_med"] - 1.2, "GA median",
            transform=ax.get_yaxis_transform(), ha="right", va="top",
            fontsize=PT_NOTE, color=STYLE["ga"]["c"])
    ax.plot(step, exact, color=cp, lw=1.2, zorder=3,
            label=r"$\pi_\psi$, exact $F$ of its configuration")
    ax.plot(step[ok], exact[ok], "^", color=cp, ms=3.6, mec="black", mew=0.4,
            ls="none", zorder=4, label="Feasible")
    ax.plot(step[~ok], exact[~ok], "x", color=RED, ms=3.8, mew=1.0,
            ls="none", zorder=4, label="Infeasible")
    ax.set_xlabel("Cross-entropy training step")
    ax.set_ylabel("Objective $F$ (analytical model)")
    ax.set_title(r"(a) Proposal GNN $\pi_\psi$, $1{,}500$ steps in $81$ s")
    ax.legend(loc="lower right", fontsize=PT_NOTE, borderpad=0.3,
              labelspacing=0.22, handlelength=1.6, handletextpad=0.4)

    # (b) evaluation network: exact F of the V = 8 it forwards
    c = TC["curve"]
    ep = np.array([r["epoch"] for r in c], float)
    mean_v = np.array([r["exact"] for r in c], float)
    best_v = np.array([r["exact_best"] for r in c], float)
    ce = STYLE["ga_gnn"]["c"]

    ax = axes[1]
    ax.axhline(TC["pool"]["oracle_topv"], color=GREY, lw=0.8, ls=(0, (1, 2)),
               zorder=2)
    ax.text(0.98, TC["pool"]["oracle_topv"] - 1.2,
            "perfect ordering of the same pool",
            transform=ax.get_yaxis_transform(), ha="right", va="top",
            fontsize=PT_NOTE, color=GREY)
    ax.fill_between(ep, mean_v, best_v, color=ce, alpha=0.12, lw=0)
    ax.plot(ep, best_v, ls=(0, (1, 1.6)), color=ce, lw=0.9, zorder=3,
            label=r"best of the $V = 8$ it forwards")
    ax.plot(ep, mean_v, "-s", color=ce, ms=3.2, mec="black", mew=0.4, zorder=4,
            label=r"mean of the $V = 8$ it forwards")
    ax.set_xlabel("Training epoch")
    ax.set_title(r"(b) Evaluation GNN $g_\phi$, $%d$ epochs in $%d$ s"
                 % (TC["epochs"], round(c[-1]["elapsed"])))
    ax.legend(loc="lower right", fontsize=PT_NOTE, borderpad=0.3,
              labelspacing=0.22, handlelength=1.6, handletextpad=0.4)

    axes[0].set_ylim(-9, 60)
    fig.tight_layout(pad=0.3)
    if save:
        fig.savefig(save, dpi=600)
        print("saved", save)
    plt.show()

    first = int(np.argmax(ok))
    print("\npi_psi  : first feasible configuration at step %d" % step[first])
    hit = next(r for r in c if r["exact"] >= 0.999 * TC["pool"]["oracle_topv"])
    print("g_phi   : within 0.1%% of a perfect ordering at epoch %d (%.0f s)"
          % (hit["epoch"], hit["elapsed"]))

fig_training()

## Fig. 4 — objective against compute

The comparison the other figures cannot make. At a common $(N_{pop}, N_{gen})$
the evaluation GNN is **not** expected to raise the objective: it swaps an exact
evaluator for an approximate one, so at an equal number of candidates examined
it can at best match the baseline, and it carries prediction error besides. What
it buys is that each candidate costs less — a common grid point costs the three
methods $12{,}426$, $1{,}070$ and $474$ exact calls.

Whether the optimiser can *use* that cheapness is a separate question, and this
is the figure that answers it: compute on the x axis, each curve read as "what
had this method reached by then", 20 seeds, no early stopping.

In [ ]:
# --- Cell 7: Fig. 4 ----------------------------------------------------------
def fig_anytime(save="fig4_anytime.png"):
    t = np.array(AT["t_grid"], float)
    thr = AT["success"]
    fig, axes = plt.subplots(1, 2, figsize=(PAGE, 2.45))

    for k in ORDER:
        st, a = STYLE[k], AT["methods"][k]["wall"]
        mu = np.array(a["mean"], float)
        sd = np.array(a["std"], float)
        su = np.array(a["success"], float)
        # Before a method's first generation there is nothing to report. The
        # run-out is NaN rather than extrapolated backwards -- extrapolating
        # there would hand free credit to whichever method starts slowest.
        ok = np.isfinite(mu)
        axes[0].fill_between(t[ok], (mu - sd)[ok], (mu + sd)[ok], color=st["c"],
                             alpha=0.10, lw=0, zorder=1)
        axes[0].plot(t[ok], mu[ok], ls=st["ls"], color=st["c"], lw=1.3,
                     zorder=3, label=LAB[k])
        axes[1].plot(t[ok], 100 * su[ok], ls=st["ls"], color=st["c"], lw=1.3,
                     zorder=3, label=LAB[k])

    ax = axes[0]
    ax.axhline(thr, color=RED, lw=0.8, ls=(0, (3, 2)), zorder=2)
    ax.text(0.02, thr - 1.5, "success threshold $F \\geq %.2f$" % thr,
            transform=ax.get_yaxis_transform(), ha="left", va="top",
            fontsize=PT_NOTE, color=RED)
    ax.set_ylabel("Objective $F$ reached by then")
    ax.set_title("(a) What each method has reached, mean of $%d$ seeds"
                 % AT["n_seed"])

    ax = axes[1]
    ax.set_ylim(-4, 108)
    ax.set_ylabel(r"Seeds reaching $F \geq %.2f$ (\%%)" % thr)
    ax.set_title("(b) How often, at the same budget")

    for ax in axes:
        ax.set_xscale("log")
        ax.set_xlabel("Wall-clock budget per solve (s)")

    fig.tight_layout(pad=0.3)
    legend_below(fig, axes[0], ncol=3, y=-0.17)
    if save:
        fig.savefig(save, dpi=600)
        print("saved", save)
    plt.show()

    # The claim the figure exists to support, checked rather than asserted.
    for axis, gk in (("wall", "t_grid"), ("evals", "e_grid")):
        g = np.array(AT[gk], float)
        a = np.array(AT["methods"]["ga_gnn"][axis]["mean"], float)
        b = np.array(AT["methods"]["ga"][axis]["mean"], float)
        c = np.array(AT["methods"]["ga_gnn_policy"][axis]["mean"], float)
        m = np.isfinite(a) & np.isfinite(b) & np.isfinite(c)
        print("  %-6s pipeline > g_phi > GA at %d of %d shared budgets "
              "(%.3g to %.3g)"
              % (axis, int(((c > a) & (a > b) & m).sum()), int(m.sum()),
                 g[m][0], g[m][-1]))

fig_anytime()

## Running time

Two costs, and they are paid at different times.

**Per solve** is what the operator pays every time a scenario arrives. Both the
wall-clock time and the number of analytical-model calls are given: wall-clock
depends on the machine, calls do not, so the calls column is the one that
transfers.

**One-off training** is paid once for all scenarios. It is reported separately
rather than amortised into a per-solve figure, because how it amortises depends
entirely on how many scenarios a deployment solves.

In [ ]:
# --- Cell 8: running-time table ----------------------------------------------
try:
    import pandas as pd
except ImportError:
    pd = None

def runtime_tables():
    # ---- per solve, common budget 120x120, main scenario, 10 seeds ----------
    rows = []
    ga_wall = QM["configs"]["ga"]["wall_median"]
    ga_call = QM["configs"]["ga"]["n_eval_median"]
    for k in ORDER:
        c = QM["configs"][k]
        rows.append({
            "Method": LAB_TXT[k], "_tex": LAB_TEX[k],
            "Wall (s)": round(c["wall_median"], 2),
            "Model calls": int(c["n_eval_median"]),
            "F best of 10": round(c["objective"], 2),
            "F median": round(c["objective_median"], 2),
            "Speed-up": "%.1fx" % (ga_wall / c["wall_median"]),
            "Calls saved": "%.1fx" % (ga_call / c["n_eval_median"]),
        })
    # the configuration actually deployed: one refinement generation, not 120
    rows.append({
        "Method": DEPLOY_TXT, "_tex": DEPLOY_TEX,
        "Wall (s)": round(float(np.median(NG1["wall"])), 2),
        "Model calls": int(np.median(NG1["eval"])),
        "F best of 10": round(float(np.max(NG1["fit"])), 2),
        "F median": round(float(np.median(NG1["fit"])), 2),
        "Speed-up": "%.1fx" % (ga_wall / float(np.median(NG1["wall"]))),
        "Calls saved": "%.1fx" % (ga_call / float(np.median(NG1["eval"]))),
    })

    # ---- per solve across the budget grid ----------------------------------
    # Index within each method's own budget list, not by position in the global
    # one: the lists have diverged since some budgets were measured for a single
    # method only. Rows are restricted to budgets all three actually ran, since
    # a row with two blanks compares nothing.
    grid = []
    for b in BM["budgets"]:
        if not all(b in BM["methods"][k]["budget"] for k in ORDER):
            continue
        r = {"Budget (Npop x Ngen)": b}
        for k in ORDER:
            e = BM["methods"][k]
            i = e["budget"].index(b)
            tag = {"ga": "GA", "ga_gnn": "+g_phi", "ga_gnn_policy": "+pi_psi"}[k]
            r["%s: s" % tag] = round(e["wall_per_run"][i], 2)
            r["%s: calls" % tag] = int(round(e["calls"][i]))
            r["%s: mean F" % tag] = round(e["best_mean"][i], 2)
        grid.append(r)

    # ---- one-off training ---------------------------------------------------
    train = [
        {"Stage": "Training-set generation (200,000 samples, 26 cores)",
         "Seconds": 159},
        {"Stage": "Evaluation GNN g_phi (%d epochs)" % TC["epochs"],
         "Seconds": round(TC["curve"][-1]["elapsed"])},
        {"Stage": "Proposal GNN pi_psi (1,500 cross-entropy steps)",
         "Seconds": 81},
    ]
    train.append({"Stage": "Total, paid once for all scenarios",
                  "Seconds": sum(r["Seconds"] for r in train)})

    def show(title, data):
        print("\n" + title)
        print("-" * len(title))
        # Columns whose name starts with "_" carry the LaTeX spelling of a
        # field for the export cell; they are not for reading.
        vis = [{k: v for k, v in r.items() if not k.startswith("_")}
               for r in data]
        if pd is not None:
            df = pd.DataFrame(vis)
            try:
                from IPython.display import display
                display(df)
            except ImportError:
                print(df.to_string(index=False))
            return df
        for r in vis:
            print("  " + "  ".join("%s=%s" % kv for kv in r.items()))

    t1 = show("Per solve, main scenario, Npop = Ngen = 120, median of 10 seeds",
              rows)
    t2 = show("Per solve across the budget grid (mean F over the five eps_1)",
              grid)
    t3 = show("One-off training cost", train)
    return rows, t1, t2, t3

ROWS, T1, T2, T3 = runtime_tables()

In [ ]:
# --- Cell 9: the same table as LaTeX, ready to paste into the manuscript -----
# Written out field by field rather than through DataFrame.to_latex: that
# helper escapes the backslashes in the method names into \textbackslash and
# prints every float at six decimals.
def latex_runtime(rows):
    L = [r"\begin{table}[!t]", r"\centering",
         r"\caption{Cost of one solve on the reference scenario, median over "
         r"ten seeds at a common budget $\Npop = \Ngen = 120$. The last row is "
         r"the configuration actually deployed. Analytical-model calls are "
         r"machine independent; wall-clock time is on one RTX~4060 and a "
         r"26-core CPU.}",
         r"\label{tab:runtime}", r"\small",
         r"\setlength{\tabcolsep}{4pt}",
         r"\begin{tabular}{lrrrrr}", r"\toprule",
         r"Configuration & Time (s) & Exact calls & $F$ best & $F$ med. "
         r"& Speed-up \\", r"\midrule"]
    for i, r in enumerate(rows):
        if i == len(rows) - 1:
            L.append(r"\midrule")
        L.append("%s & $%.2f$ & $%s$ & $%.2f$ & $%.2f$ & $%s$ \\\\"
                 % (r["_tex"], r["Wall (s)"], "{:,}".format(r["Model calls"]),
                    r["F best of 10"], r["F median"],
                    r["Speed-up"].replace("x", r"\times")))
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(L)

print(latex_runtime(ROWS))